# Linux Day 2026 - vLLM test and benchmarks

In this notebook, you'll learn how to:
1. Launch a **vLLM inference server** serving a real model
2. Query it using the OpenAI-compatible API
3. **Explore model behavior** with logprobs and sampling parameters
4. Compare **vLLM vs Ollama** throughput as concurrency increases
5. Compare **prefix caching** on and off by time to first token and output tokens per request


## Step 1: Start Your vLLM Server

**How to serve a model with vLLM?**

To serve your model with vLLM, you'd need to run this command in the terminal. 

```bash
vllm serve Qwen/Qwen3-1.7B --dtype=bfloat16 --max-model-len 4096
```

- **`vllm serve`**: launches vLLM's built-in inference server. It loads the model weights into the engine (with PagedAttention, continuous batching, and prefix caching enabled by default) and exposes it over HTTP on port 8000.
- **`Qwen/Qwen3-1.7B`**: the model identifier on the [Hugging Face Hub](https://huggingface.co/Qwen/Qwen3-1.7B). On first run, vLLM downloads the weights, tokenizer, and config from Hugging Face into the local cache (`~/.cache/huggingface/hub`), then loads them into memory. Subsequent runs reuse the cached files.
- **`--dtype=bfloat16`**: loads the weights in BF16 precision. 
- **`--max-model-len 4096`**: caps the context window (prompt + generation) at 4096 tokens. 


vLLM wraps the model in an **OpenAI-compatible HTTP API**. This notebook calls `/v1/models` and `/v1/chat/completions`.


Check that the server is up. The first launch can take 1 or 2 minutes while the weights load. The next cell waits up to 5 minutes.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, math, sys

VLLM_URL = "http://localhost:8000"
os.makedirs("outputs", exist_ok=True)

print("Waiting for vLLM server...")
for attempt in range(60):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            data = r.json().get("data") or []
            if data:
                MODEL = data[0]["id"]
                break
    except requests.RequestException:
        pass
    time.sleep(5)
    if attempt % 6 == 5:
        print(f"  Still waiting... ({(attempt + 1) * 5}s elapsed)")
else:
    raise RuntimeError(
        "vLLM server not reachable after 5 minutes."
    )

print(f"Connected to {VLLM_URL} — model: {MODEL}")


## Your First Local LLM Request

vLLM exposes an OpenAI-compatible API, so we use the standard `openai` Python client.

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=120, max_retries=0)


Qwen3 supports a *thinking mode* that generates chain-of-thought reasoning before answering. We disable it with `enable_thinking: False` to keep responses short and predictable.

`top_k=20` keeps only the 20 most likely next tokens. `top_p=0.8` then keeps the best of those until their probabilities add up to 80%.


In [ ]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "Tell me a short joke"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20, 
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {resp.usage.total_tokens} total")

## Exploring Model Behavior

Beyond simple chat, the vLLM API lets you inspect model internals and control generation. For example, **Logprobs:** allows you to see the model's confidence in each token it generates, plus the alternatives it considered

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of Italy is"}],
    max_tokens=15,
    temperature=0.0,
    logprobs=True,
    top_logprobs=5,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(f"Response: {resp.choices[0].message.content.strip()}\n")
print("Token-by-token probabilities:\n")

for tok in resp.choices[0].logprobs.content[:8]:
    print(f"  Chosen: '{tok.token}'  (logprob {tok.logprob:.2f})")
    if tok.top_logprobs:
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "\u2588" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  '{alt.token}'")
    print()

<br/>

---
<br/>

## vLLM vs Ollama under Concurrency

vLLM and Ollama can serve the same model. The sweep is a fair comparison only when both do the same work, can run the same number of requests at once, and each gets the GPU in turn.

- **Same weights.** vLLM loads the checkpoint with `--dtype=bfloat16`. Ollama's matching file is the `fp16` tag, such as `qwen3:1.7b-fp16`. The unsuffixed tag is a 4-bit quant. Set `OLLAMA_MODEL` to the fp16 tag.
- **Same concurrency budget.** `OLLAMA_NUM_PARALLEL` defaults to 1, so extra callers wait even when the GPU is idle. Start Ollama with as many slots as the highest level below, and with the same context the vLLM server reports:

```bash
OLLAMA_NUM_PARALLEL=32 OLLAMA_CONTEXT_LENGTH=4096 ollama serve
```

Thirty-two slots need more GPU memory. If the model cannot load, stop other GPU processes, or lower both `OLLAMA_NUM_PARALLEL` and `CONCURRENCY_LEVELS` to the same maximum.

- **One engine on the GPU at a time.** The vLLM cell runs first. Stop `vllm serve` before the Ollama cell. vLLM keeps the model loaded until that process exits.
- **Same questions.** `data/benchmark_questions.json` supplies a different question per request, in the same order on both engines, so a repeated prompt does not let prefix caching skip the prefill.
- **Same decoding limits.** Greedy sampling, thinking off, and at most 64 new tokens.

The charts follow the [Red Hat Ollama vs vLLM benchmark](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking): rising concurrency, then output tokens per second and requests per second. vLLM is on port 8000 and Ollama is on port 11434.


In [ ]:
import requests
from openai import OpenAI

OLLAMA_URL = "http://localhost:11434"
# Set this to an Ollama tag to skip auto-matching, for example "qwen3:1.7b-fp16".
OLLAMA_MODEL = "qwen3:1.7b-fp16"


def list_model_ids(base_url):
    """Return model ids from an OpenAI-compatible /v1/models endpoint."""
    try:
        response = requests.get(f"{base_url}/v1/models", timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not reach {base_url}/v1/models. "
            "Start that server, then rerun this cell."
        ) from exc
    payload = response.json()
    data = payload.get("data") if isinstance(payload, dict) else None
    return [
        item["id"]
        for item in data or []
        if isinstance(item, dict) and item.get("id")
    ]


def list_ollama_models(base_url):
    """Prefer Ollama's OpenAI endpoint, and fall back to /api/tags for names.

    Some Ollama builds return {"data": null} from /v1/models instead of an empty
    list. That is not a connection error, so fall through to /api/tags.
    """
    def names_from(items, key):
        names = []
        for item in items or []:
            if isinstance(item, dict):
                name = item.get(key)
            elif isinstance(item, str):
                name = item
            else:
                name = None
            if name:
                names.append(name)
        return names

    try:
        response = requests.get(f"{base_url}/v1/models", timeout=10)
        if response.status_code == 200:
            payload = response.json()
            data = payload.get("data") if isinstance(payload, dict) else None
            ids = names_from(data, "id")
            if ids:
                return ids
    except (requests.RequestException, ValueError):
        pass
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not reach Ollama at {base_url}. "
            "Expected /v1/models (port 11434 by default)."
        ) from exc
    payload = response.json()
    models = payload.get("models") if isinstance(payload, dict) else None
    return names_from(models, "name")


def _norm(name):
    return "".join(ch for ch in name.lower() if ch.isalnum())


def _full_precision(names):
    return [name for name in names if "fp16" in name.lower() or "bf16" in name.lower()]


def choose_ollama_model(vllm_model, ollama_models, override):
    if not ollama_models:
        raise RuntimeError(
            "Ollama has no models. Pull the fp16 tag that matches vLLM, "
            "for example: ollama pull qwen3:1.7b-fp16"
        )
    if override:
        if override not in ollama_models:
            raise RuntimeError(
                f"OLLAMA_MODEL={override!r} is not loaded. Ollama has: {ollama_models}"
            )
        return override

    target = _norm(vllm_model.split("/")[-1])
    related = [
        name for name in ollama_models
        if target in _norm(name) or _norm(name) in target
    ]
    full = _full_precision(related)
    if full:
        return min(full, key=len)
    exact = [name for name in related if _norm(name) == target]
    if exact:
        return exact[0]
    if related:
        return min(related, key=lambda name: abs(len(_norm(name)) - len(target)))
    if len(ollama_models) == 1:
        print(f"No name match for {vllm_model}. Using the only Ollama model: {ollama_models[0]}")
        return ollama_models[0]

    raise RuntimeError(
        "Could not match the vLLM model to an Ollama tag. "
        f"vLLM is serving {vllm_model}. Ollama has {ollama_models}. "
        "Set OLLAMA_MODEL to one of those tags and rerun this cell."
    )


vllm_models = list_model_ids(VLLM_URL)
ollama_models = list_ollama_models(OLLAMA_URL)
print("vLLM models:  ", vllm_models)
print("Ollama models:", ollama_models)

VLLM_MODEL = MODEL
OLLAMA_MODEL = choose_ollama_model(VLLM_MODEL, ollama_models, OLLAMA_MODEL)
stem = _norm(VLLM_MODEL.split("/")[-1])
chosen = _norm(OLLAMA_MODEL)
if stem not in chosen and chosen not in stem:
    print(
        "These names do not look like the same model. "
        "Set OLLAMA_MODEL above if this is the wrong tag."
    )

vllm_client = OpenAI(
    base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=600, max_retries=0,
)
print()
print(f"Comparing vLLM '{VLLM_MODEL}' with Ollama '{OLLAMA_MODEL}'")
if "fp16" not in OLLAMA_MODEL.lower() and "bf16" not in OLLAMA_MODEL.lower():
    print(
        "This Ollama tag is not an fp16 or bf16 file. The unsuffixed tag is usually "
        "a 4-bit quant, so it is not the same weights as vLLM's bfloat16 server. "
        "Set OLLAMA_MODEL to the fp16 tag and rerun this cell."
    )


In [ ]:
import concurrent.futures

from openai import APIStatusError

# Same questions, same order, one engine per cell. vLLM runs here; Ollama runs next.
DATA_PATH = "data/benchmark_questions.json"
MAX_TOKENS = 64
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32]
REQUESTS_PER_USER = 2  # each in-flight slot sends this many different questions back to back
WARMUP_PROMPT = "Reply with the single word ready."
RESULTS_PATH = "outputs/vllm_vs_ollama.json"


def server_context_length():
    """Context window of the running vLLM server. Ollama is pinned to the same value."""
    try:
        response = requests.get(f"{VLLM_URL}/v1/models", timeout=10)
        response.raise_for_status()
        data = response.json().get("data") or []
        length = data[0].get("max_model_len") if data else None
        if length:
            return int(length)
    except (requests.RequestException, ValueError, TypeError, IndexError, AttributeError):
        pass
    return 4096


CONTEXT_LENGTH = server_context_length()


def load_questions(path):
    with open(path) as f:
        payload = json.load(f)
    questions = payload["questions"] if isinstance(payload, dict) else payload
    cleaned = []
    seen = set()
    for item in questions:
        text = item.strip() if isinstance(item, str) else ""
        if not text or text in seen:
            raise RuntimeError(f"{path} has a blank or repeated question.")
        seen.add(text)
        cleaned.append(text)
    needed = sum(CONCURRENCY_LEVELS) * REQUESTS_PER_USER
    if len(cleaned) < needed:
        raise RuntimeError(
            f"{path} has {len(cleaned)} questions but this sweep needs {needed}."
        )
    return cleaned


def read_prefix_counters():
    try:
        text = requests.get(f"{VLLM_URL}/metrics", timeout=5).text
    except requests.RequestException:
        return None
    totals = {"hits": 0.0, "queries": 0.0}
    found = False
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        key = line.split("{", 1)[0].split()[0]
        if key.endswith("_created"):
            continue
        if key.endswith("prefix_cache_hits") or key.endswith("prefix_cache_hits_total"):
            bucket = "hits"
        elif key.endswith("prefix_cache_queries") or key.endswith("prefix_cache_queries_total"):
            bucket = "queries"
        else:
            continue
        try:
            totals[bucket] += float(line.split()[-1])
            found = True
        except ValueError:
            continue
    return totals if found else None


def call_vllm(model, prompt, extra_holder):
    kwargs = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": MAX_TOKENS,
        "temperature": 0,
        "top_p": 1,
    }
    if extra_holder[0]:
        kwargs["extra_body"] = extra_holder[0]
    started = time.perf_counter()
    try:
        response = vllm_client.chat.completions.create(**kwargs)
    except APIStatusError as exc:
        if extra_holder[0] and exc.status_code in (400, 422):
            print("  vLLM rejected the thinking flag; retrying without it")
            extra_holder[0] = None
            return call_vllm(model, prompt, extra_holder)
        raise
    elapsed = time.perf_counter() - started
    usage = response.usage
    if usage is None or usage.completion_tokens is None:
        raise RuntimeError(f"{model} did not return a completion token count.")
    return elapsed, usage.completion_tokens, usage.prompt_tokens or 0, 0


def call_ollama(model, prompt, think_holder):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "num_ctx": CONTEXT_LENGTH,
            "num_predict": MAX_TOKENS,
            "temperature": 0,
            "top_p": 1,
        },
    }
    if think_holder[0]:
        payload["think"] = False
    started = time.perf_counter()
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
    elapsed = time.perf_counter() - started
    if response.status_code in (400, 422) and think_holder[0]:
        print("  Ollama rejected the thinking flag; retrying without it")
        think_holder[0] = False
        return call_ollama(model, prompt, think_holder)
    if response.status_code >= 400:
        raise RuntimeError(
            f"Ollama /api/chat failed ({response.status_code}): {response.text[:300]}"
        )
    body = response.json()
    completion = body.get("eval_count")
    if completion is None:
        raise RuntimeError(f"{model} did not return an output token count.")
    return (
        elapsed,
        completion,
        body.get("prompt_eval_count") or 0,
        body.get("prompt_eval_cached_count") or 0,
    )


def benchmark_engine(engine, model, questions):
    """Keep `concurrency` requests in flight. Each request uses a different question."""
    print()
    print(f"{engine}  model={model}")
    extra_holder = [{"chat_template_kwargs": {"enable_thinking": False}}]
    think_holder = [True]
    if engine == "vLLM":
        warmup = call_vllm(model, WARMUP_PROMPT, extra_holder)
    else:
        warmup = call_ollama(model, WARMUP_PROMPT, think_holder)
    print(f"  warmup: {warmup[0]:.2f}s, {warmup[1]} completion tokens")
    print(
        "  wall is how many seconds this level took, from the first send "
        "until the last response. overlapped is how many requests were in "
        "flight together: add up each request's own time and divide by wall. "
        "It stays near the concurrency number when the engine runs them side by side."
    )

    def one_user(prompts):
        totals = [0, 0, 0]  # completion, prompt, cached
        latencies = []
        for prompt in prompts:
            if engine == "vLLM":
                elapsed, completion, prompt_count, cached = call_vllm(
                    model, prompt, extra_holder
                )
            else:
                elapsed, completion, prompt_count, cached = call_ollama(
                    model, prompt, think_holder
                )
            latencies.append(elapsed)
            totals[0] += completion
            totals[1] += prompt_count
            totals[2] += cached
        return latencies, totals

    rows = []
    offset = 0
    warned = False
    for concurrency in CONCURRENCY_LEVELS:
        n_requests = concurrency * REQUESTS_PER_USER
        batch = questions[offset:offset + n_requests]
        offset += n_requests
        groups = [
            batch[index * REQUESTS_PER_USER:(index + 1) * REQUESTS_PER_USER]
            for index in range(concurrency)
        ]
        print(f"  concurrency {concurrency}: {n_requests} different questions ...", flush=True)
        started = time.perf_counter()
        with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
            parts = list(pool.map(one_user, groups))
        wall_s = time.perf_counter() - started

        latencies = [dt for lat, _ in parts for dt in lat]
        output_tokens = sum(totals[0] for _, totals in parts)
        prompt_tokens = sum(totals[1] for _, totals in parts)
        cached_tokens = sum(totals[2] for _, totals in parts)
        achieved = sum(latencies) / wall_s if wall_s else 0.0
        row = {
            "engine": engine,
            "model": model,
            "concurrency": concurrency,
            "requests": n_requests,
            "wall_s": round(wall_s, 3),
            "output_tokens": output_tokens,
            "prompt_tokens": prompt_tokens,
            "cached_prompt_tokens": cached_tokens,
            "tps": output_tokens / wall_s,
            "rps": n_requests / wall_s,
            "avg_latency_s": sum(latencies) / len(latencies),
            "achieved_parallel": achieved,
        }
        rows.append(row)
        print(
            f"    {row['tps']:.1f} tok/s   {row['rps']:.2f} req/s   "
            f"wall {wall_s:.1f}s   overlapped {achieved:.1f}"
        )
        if (
            engine == "Ollama"
            and not warned
            and concurrency >= 2
            and achieved < 0.6 * concurrency
        ):
            warned = True
            print(
                "    These requests did not overlap. OLLAMA_NUM_PARALLEL is probably "
                "still 1, so later levels are a queue, not a fair comparison."
            )
    return rows, warmup[1]


def save_results(rows):
    os.makedirs("outputs", exist_ok=True)
    with open(RESULTS_PATH, "w") as f:
        json.dump(rows, f, indent=2)


questions = load_questions(DATA_PATH)
needed = sum(CONCURRENCY_LEVELS) * REQUESTS_PER_USER
print(
    f"Loaded {len(questions)} questions; using {needed}. "
    f"Context matched to vLLM max_model_len {CONTEXT_LENGTH}."
)
print("Running vLLM only. Stop that server before the Ollama cell so it can use the GPU.")

prefix_before = read_prefix_counters()
vllm_rows, vllm_warmup = benchmark_engine("vLLM", VLLM_MODEL, questions)
prefix_after = read_prefix_counters()
warmup_tokens = {"vLLM": vllm_warmup}
bench_results = vllm_rows
save_results(bench_results)

if prefix_before and prefix_after:
    hits = prefix_after["hits"] - prefix_before["hits"]
    queries = prefix_after["queries"] - prefix_before["queries"]
    print(f"vLLM prefix-cache queries: {queries:.0f}, hits {hits:.0f}.")
    if queries > 0:
        print(
            "Hits are mostly the short chat header shared by every request. "
            "The questions themselves differ, so the long prefix is not reused."
        )
print(f"Saved {RESULTS_PATH}")


**Stop vLLM before the next cell.**

vLLM keeps the model loaded on the GPU until that process exits. A single GPU then has almost no memory left for Ollama, and the Ollama half measures CPU spill rather than the engine. Stop `vllm serve`, leave Ollama running with `OLLAMA_NUM_PARALLEL` set to 32, then run the next cell.


In [ ]:
# Stop vLLM before running this cell. Ollama must already be serving with
# OLLAMA_NUM_PARALLEL equal to the highest concurrency level.
if "benchmark_engine" not in globals() or "questions" not in globals():
    raise RuntimeError("Run the vLLM benchmark cell in this kernel first.")

with open(RESULTS_PATH) as f:
    bench_results = [row for row in json.load(f) if row["engine"] == "vLLM"]
if not bench_results:
    raise RuntimeError(f"{RESULTS_PATH} has no vLLM rows. Run the previous cell first.")

ollama_rows, ollama_warmup = benchmark_engine("Ollama", OLLAMA_MODEL, questions)
warmup_tokens["Ollama"] = ollama_warmup
bench_results.extend(ollama_rows)
save_results(bench_results)

v_tok = warmup_tokens["vLLM"]
o_tok = warmup_tokens["Ollama"]
print()
print(f"Warmup completion tokens — vLLM: {v_tok}, Ollama: {o_tok}")
if min(v_tok, o_tok) > 0 and max(v_tok, o_tok) > 2 * min(v_tok, o_tok):
    print(
        "Those lengths differ by more than 2x, so one engine is generating "
        "much more text per request. Throughput is then not the same amount of work."
    )

top = ollama_rows[-1]
fair_slots = max(CONCURRENCY_LEVELS)
if top["achieved_parallel"] < 0.6 * top["concurrency"]:
    print(
        f"Ollama overlapped about {top['achieved_parallel']:.1f} requests "
        f"at concurrency {top['concurrency']}."
    )
    print(
        "Restart it with "
        f"OLLAMA_NUM_PARALLEL={fair_slots} OLLAMA_CONTEXT_LENGTH={CONTEXT_LENGTH} "
        "ollama serve"
    )
else:
    print(
        f"Ollama overlapped about {top['achieved_parallel']:.1f} requests "
        f"at concurrency {top['concurrency']}."
    )

cached = sum(row["cached_prompt_tokens"] for row in ollama_rows)
prompt_total = sum(row["prompt_tokens"] for row in ollama_rows)
if prompt_total:
    cached_pct = 100.0 * cached / prompt_total
    print(f"Ollama prompt tokens read from cache: {cached} / {prompt_total} ({cached_pct:.0f}%).")
    if cached_pct > 50:
        print("More than half of each Ollama prompt was cached, so this run still reuses a prefix.")
print(f"Saved {RESULTS_PATH}")


In [ ]:
import json
import os

import matplotlib.pyplot as plt

if "bench_results" not in globals():
    with open("outputs/vllm_vs_ollama.json") as f:
        bench_results = json.load(f)

ENGINES = ["vLLM", "Ollama"]
COLORS = {"vLLM": "#CC0000", "Ollama": "#1B4F9C"}
MARKERS = {"vLLM": "o", "Ollama": "s"}


def rows_for(engine):
    return sorted(
        (row for row in bench_results if row["engine"] == engine),
        key=lambda row: row["concurrency"],
    )


missing = [engine for engine in ENGINES if not rows_for(engine)]
if missing:
    raise RuntimeError(
        f"No rows for {', '.join(missing)}. Run the vLLM cell and the Ollama cell first."
    )

print(
    f"{'Engine':<8} {'Concurrency':>12} {'Tokens/s':>10} "
    f"{'Requests/s':>12} {'Latency (s)':>12} {'Overlap':>8}"
)
print("-" * 66)
for engine in ENGINES:
    for row in rows_for(engine):
        print(
            f"{row['engine']:<8} {row['concurrency']:>12} "
            f"{row['tps']:>10.1f} {row['rps']:>12.2f} "
            f"{row['avg_latency_s']:>12.2f} {row['achieved_parallel']:>8.1f}"
        )

v_rows = rows_for("vLLM")
o_rows = rows_for("Ollama")
v_tps, o_tps = v_rows[-1]["tps"], o_rows[-1]["tps"]
hi = v_rows[-1]["concurrency"]
if o_tps > 0:
    print(
        f"At {hi} concurrent requests: "
        f"vLLM {v_tps:.1f} tok/s, Ollama {o_tps:.1f} tok/s "
        f"(vLLM / Ollama = {v_tps / o_tps:.2f})."
    )

with plt.rc_context({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
    "font.size": 11,
    "axes.axisbelow": True,
}):
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 7.4), sharex=True, constrained_layout=True)
    fig.suptitle("vLLM vs Ollama — distinct questions, rising concurrency", fontsize=14)
    panels = [
        ("tps", "Output tokens per second", "Figure 1. Output token throughput"),
        ("rps", "Requests per second", "Figure 2. Request throughput"),
    ]
    levels = [row["concurrency"] for row in v_rows]
    for ax, (key, ylabel, title) in zip(axes, panels):
        for engine in ENGINES:
            rows = rows_for(engine)
            ax.plot(
                [row["concurrency"] for row in rows],
                [row[key] for row in rows],
                label=engine,
                color=COLORS[engine],
                marker=MARKERS[engine],
                linewidth=2.25,
                markersize=8,
            )
        ax.set_ylabel(ylabel)
        ax.set_title(title, loc="left", fontsize=12)
        ax.set_xticks(levels)
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False)
    axes[-1].set_xlabel("Concurrent requests")
    os.makedirs("outputs", exist_ok=True)
    fig.savefig("outputs/vllm_vs_ollama_throughput.png", dpi=140, bbox_inches="tight")
    plt.show()

print("Saved outputs/vllm_vs_ollama_throughput.png")


**How to read the charts**

Both figures use the same concurrency levels and the same questions. vLLM ran while it owned the GPU. Ollama ran after that process was stopped.

**Tokens per second** is output tokens divided by wall-clock time. **Requests per second** is completed calls divided by that same time. A call that finishes in under a second produces more than one request per second on a single slot, so Figure 2 can sit above the concurrency numbers. Requests per second multiplied by average latency should land near that row's **Overlap** column. Overlap is the latency sum divided by wall time: the number of requests that were actually in flight together.

A line that climbs is the engine doing more work as more requests arrive together. A line that flattens has hit its parallelism limit.


<br/>

---
<br/>

## Prefix Caching

Many applications send the same long **system prompt** with every request. **Prefix caching** is on by default: the first request builds the KV cache for that shared prefix, and later requests reuse it. Only the new question and the short answer are new work, so those later requests finish sooner. **Output tokens/s** in this section is answer tokens divided by the whole request, so a shorter prefill raises that rate.

This machine can run only one vLLM server at a time, so the two settings are measured one after the other on port 8000. The benchmark uses a long repeated system prompt and caps each answer at a few tokens, so prefill is most of the request. That is the work prefix caching skips after the first call, which shows up as a lower time to first token and higher output tokens/s.

Stop Ollama before this step. The concurrency run left the model loaded for 30 minutes, and vLLM needs that GPU memory. Unload it with `ollama stop qwen3:1.7b-fp16`.

**Step 1.** Start the server with prefix caching off, wait until it is ready, then run the next cell:

```bash
vllm serve Qwen/Qwen3-1.7B --dtype=bfloat16 --max-model-len 4096 --no-enable-prefix-caching
```


In [ ]:
import json
import os
import time

import requests
from openai import OpenAI

# Run this while vLLM was started with --no-enable-prefix-caching.
# It only talks to the server already on port 8000 and saves the timings.
if "VLLM_URL" not in globals():
    VLLM_URL = "http://localhost:8000"

SHARED_BLOCK = (
    "You are a helpful AI teaching assistant for a course on "
    "LLM optimization. Cover quantization, the KV cache, continuous "
    "batching, PagedAttention, and model serving. "
    "Answer in one short sentence.\n"
)
# Long shared prefix, short answer: prefill is most of the request.
# Lower PREFIX_REPEATS if a request exceeds the server context length.
PREFIX_REPEATS = 60
SYSTEM_PROMPT = SHARED_BLOCK * PREFIX_REPEATS
MAX_TOKENS = 8
OFF = "Prefix caching off"
OFF_PATH = "outputs/prefix_caching_off.json"
EXTRA = {"chat_template_kwargs": {"enable_thinking": False}}

questions = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]


def active_model():
    try:
        response = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"No vLLM server at {VLLM_URL}. "
            "Start it with --no-enable-prefix-caching, then rerun this cell."
        ) from exc
    return response.json()["data"][0]["id"]


def read_counters(base_url):
    """Sum prefix-cache hit and query counters. Ignore Prometheus *_created gauges."""
    try:
        text = requests.get(f"{base_url}/metrics", timeout=10).text
    except requests.RequestException:
        return None
    totals = {"hits": 0.0, "queries": 0.0}
    found = False
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        key = line.split("{", 1)[0].split()[0]
        if key.endswith("prefix_cache_hits") or key.endswith("prefix_cache_hits_total"):
            bucket = "hits"
        elif key.endswith("prefix_cache_queries") or key.endswith("prefix_cache_queries_total"):
            bucket = "queries"
        else:
            continue
        try:
            totals[bucket] += float(line.split()[-1])
            found = True
        except ValueError:
            continue
    return totals if found else None


def measure_one(chat_client, model_name, mode, request_idx, question):
    """Stream so time-to-first-token is the prefill, separate from the short answer."""
    started = time.perf_counter()
    ttft = None
    usage = None
    stream = chat_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=MAX_TOKENS,
        temperature=0,
        top_p=1,
        stream=True,
        stream_options={"include_usage": True},
        extra_body=EXTRA,
    )
    for chunk in stream:
        choice = chunk.choices[0] if chunk.choices else None
        if (
            ttft is None
            and choice is not None
            and choice.delta is not None
            and choice.delta.content
        ):
            ttft = time.perf_counter() - started
        if getattr(chunk, "usage", None) is not None:
            usage = chunk.usage
    elapsed = time.perf_counter() - started
    if ttft is None:
        ttft = elapsed
    if usage is None or usage.completion_tokens is None:
        raise RuntimeError(f"{mode} did not return a completion token count.")
    completion = usage.completion_tokens
    return {
        "mode": mode,
        "request": request_idx,
        "question": question,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": completion,
        "seconds": elapsed,
        "ttft_s": ttft,
        "tokens_per_s": completion / elapsed if elapsed else 0.0,
    }


def warmup(chat_client, model_name):
    chat_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=8,
        temperature=0,
        extra_body=EXTRA,
    )


def run_mode(mode, chat_client, model_name, base_url):
    print(mode, flush=True)
    print("  warmup...", flush=True)
    warmup(chat_client, model_name)
    before = read_counters(base_url)
    rows = []
    for i, question in enumerate(questions, start=1):
        row = measure_one(chat_client, model_name, mode, i, question)
        rows.append(row)
        prompt = f"  prompt {row['prompt_tokens']} tok" if i == 1 else ""
        print(
            f"  [{i}] {question:<32} ttft {row['ttft_s']:.2f}s  "
            f"{row['seconds']:.2f}s total  "
            f"{row['completion_tokens']} tok  {row['tokens_per_s']:.1f} tok/s{prompt}",
            flush=True,
        )
    after = read_counters(base_url)
    hits = None
    if before and after:
        hits = after["hits"] - before["hits"]
        queries = after["queries"] - before["queries"]
        print(f"  prefix-cache queries +{queries:.0f}, hits +{hits:.0f}")
    else:
        print("  prefix-cache counters were not on /metrics")
    return rows, hits


model_name = active_model()
bench_client = OpenAI(
    base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=300, max_retries=0,
)
print(f"Server {VLLM_URL}  model {model_name}")
print("Expect this process to be the one started with --no-enable-prefix-caching.")

off_rows, off_hits = run_mode(OFF, bench_client, model_name, VLLM_URL)
os.makedirs("outputs", exist_ok=True)
with open(OFF_PATH, "w") as f:
    json.dump(
        {
            "mode": OFF,
            "model": model_name,
            "system_prompt": SYSTEM_PROMPT,
            "questions": questions,
            "max_tokens": MAX_TOKENS,
            "hits": off_hits,
            "rows": off_rows,
        },
        f,
        indent=2,
    )

steady_tps = [row["tokens_per_s"] for row in off_rows if row["request"] > 1]
steady_ttft = [row["ttft_s"] for row in off_rows if row["request"] > 1]
print()
if steady_ttft:
    last = off_rows[-1]["request"]
    print(f"Requests 2 to {last} time to first token: {sum(steady_ttft) / len(steady_ttft):.2f}s")
    print(f"Requests 2 to {last} output tokens/s: {sum(steady_tps) / len(steady_tps):.1f}")
    if steady_ttft and off_rows[0]["ttft_s"] > 0 and (
        sum(steady_ttft) / len(steady_ttft) < 0.5 * off_rows[0]["ttft_s"]
    ):
        print(
            "Requests after the first were much faster. "
            "This server is still caching the prefix. "
            "Restart it with --no-enable-prefix-caching and run this cell again."
        )
if off_hits is not None and off_hits > 0:
    print(
        "This server recorded prefix-cache hits. "
        "Restart it with --no-enable-prefix-caching and run this cell again."
    )
print(f"Saved {OFF_PATH}")
print()
print("Stop this server. Start it again with prefix caching on (the default serve command).")
print("When port 8000 is ready, run the next code cell.")


**Step 2.** Stop the server from step 1. Start the **same model** again with prefix caching on (leave out `--no-enable-prefix-caching`):

```bash
vllm serve Qwen/Qwen3-1.7B --dtype=bfloat16 --max-model-len 4096
```

Wait until port 8000 is ready, then run the next cell. It repeats the same system prompt and the same questions, then plots both runs. The kernel can stay up while you restart vLLM. The next cell also works after a kernel restart because it reads `outputs/prefix_caching_off.json`.


In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import requests
from openai import OpenAI

# Run this after stopping the no-cache server and starting vLLM with prefix caching on.
if "VLLM_URL" not in globals():
    VLLM_URL = "http://localhost:8000"

ON = "Prefix caching on"
OFF = "Prefix caching off"
OFF_PATH = "outputs/prefix_caching_off.json"
EXTRA = {"chat_template_kwargs": {"enable_thinking": False}}

if not os.path.exists(OFF_PATH):
    raise RuntimeError(
        "Missing outputs/prefix_caching_off.json. "
        "Run the previous code cell while the server is started with "
        "--no-enable-prefix-caching."
    )
with open(OFF_PATH) as f:
    saved = json.load(f)

SYSTEM_PROMPT = saved["system_prompt"]
questions = saved["questions"]
MAX_TOKENS = saved["max_tokens"]
off_rows = saved["rows"]
if not off_rows or "ttft_s" not in off_rows[0]:
    raise RuntimeError(
        "The saved run has no time-to-first-token. Re-run the previous code cell "
        "while the server is started with --no-enable-prefix-caching."
    )
off_hits = saved.get("hits")
saved_model = saved.get("model")


def active_model():
    try:
        response = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"No vLLM server at {VLLM_URL}. "
            "Start it with prefix caching on, then rerun this cell."
        ) from exc
    return response.json()["data"][0]["id"]


def read_counters(base_url):
    try:
        text = requests.get(f"{base_url}/metrics", timeout=10).text
    except requests.RequestException:
        return None
    totals = {"hits": 0.0, "queries": 0.0}
    found = False
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        key = line.split("{", 1)[0].split()[0]
        if key.endswith("prefix_cache_hits") or key.endswith("prefix_cache_hits_total"):
            bucket = "hits"
        elif key.endswith("prefix_cache_queries") or key.endswith("prefix_cache_queries_total"):
            bucket = "queries"
        else:
            continue
        try:
            totals[bucket] += float(line.split()[-1])
            found = True
        except ValueError:
            continue
    return totals if found else None


def measure_one(chat_client, model_name, mode, request_idx, question):
    """Stream so time-to-first-token is the prefill, separate from the short answer."""
    started = time.perf_counter()
    ttft = None
    usage = None
    stream = chat_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=MAX_TOKENS,
        temperature=0,
        top_p=1,
        stream=True,
        stream_options={"include_usage": True},
        extra_body=EXTRA,
    )
    for chunk in stream:
        choice = chunk.choices[0] if chunk.choices else None
        if (
            ttft is None
            and choice is not None
            and choice.delta is not None
            and choice.delta.content
        ):
            ttft = time.perf_counter() - started
        if getattr(chunk, "usage", None) is not None:
            usage = chunk.usage
    elapsed = time.perf_counter() - started
    if ttft is None:
        ttft = elapsed
    if usage is None or usage.completion_tokens is None:
        raise RuntimeError(f"{mode} did not return a completion token count.")
    completion = usage.completion_tokens
    return {
        "mode": mode,
        "request": request_idx,
        "question": question,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": completion,
        "seconds": elapsed,
        "ttft_s": ttft,
        "tokens_per_s": completion / elapsed if elapsed else 0.0,
    }


def warmup(chat_client, model_name):
    chat_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=8,
        temperature=0,
        extra_body=EXTRA,
    )


def run_mode(mode, chat_client, model_name, base_url):
    print(mode, flush=True)
    print("  warmup...", flush=True)
    warmup(chat_client, model_name)
    before = read_counters(base_url)
    rows = []
    for i, question in enumerate(questions, start=1):
        row = measure_one(chat_client, model_name, mode, i, question)
        rows.append(row)
        prompt = f"  prompt {row['prompt_tokens']} tok" if i == 1 else ""
        print(
            f"  [{i}] {question:<32} ttft {row['ttft_s']:.2f}s  "
            f"{row['seconds']:.2f}s total  "
            f"{row['completion_tokens']} tok  {row['tokens_per_s']:.1f} tok/s{prompt}",
            flush=True,
        )
    after = read_counters(base_url)
    hits = None
    if before and after:
        hits = after["hits"] - before["hits"]
        queries = after["queries"] - before["queries"]
        print(f"  prefix-cache queries +{queries:.0f}, hits +{hits:.0f}")
    else:
        print("  prefix-cache counters were not on /metrics")
    return rows, hits


def steady(rows, key):
    picked = [row[key] for row in rows if row["request"] > 1]
    return sum(picked) / len(picked)


model_name = active_model()
if saved_model and saved_model != model_name:
    raise RuntimeError(
        f"The saved run used {saved_model}, but this server is {model_name}. "
        "Start the same model as step 1, with prefix caching on, then rerun this cell."
    )
if not saved_model:
    raise RuntimeError(
        "The saved run does not record which model it used. "
        "Re-run the previous code cell on the server started with "
        "--no-enable-prefix-caching, then run this cell again."
    )
bench_client = OpenAI(
    base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=300, max_retries=0,
)
print(f"Server {VLLM_URL}  model {model_name}")
print("Expect this process to be the one started with prefix caching on.")

on_rows, on_hits = run_mode(ON, bench_client, model_name, VLLM_URL)
prefix_rows = off_rows + on_rows
os.makedirs("outputs", exist_ok=True)
with open("outputs/prefix_caching.json", "w") as f:
    json.dump(prefix_rows, f, indent=2)

off_ttft = steady(off_rows, "ttft_s")
on_ttft = steady(on_rows, "ttft_s")
off_tps = steady(off_rows, "tokens_per_s")
on_tps = steady(on_rows, "tokens_per_s")
last = on_rows[-1]["request"]
print()
print(
    "Output tokens/s is answer tokens divided by the whole request, including prefill."
)
ttft_line = f"Requests 2 to {last} time to first token: off {off_ttft:.2f}s, on {on_ttft:.2f}s"
if on_ttft > 0 and off_ttft > on_ttft:
    ttft_line += f" ({off_ttft / on_ttft:.1f}x sooner with caching)"
print(ttft_line)
tps_line = f"Requests 2 to {last} output tokens/s:     off {off_tps:.1f}, on {on_tps:.1f}"
if off_tps > 0 and on_tps > off_tps:
    tps_line += f" ({on_tps / off_tps:.1f}x with caching)"
print(tps_line)

off_prompt = off_rows[0].get("prompt_tokens") or 0
on_prompt = on_rows[0].get("prompt_tokens") or 0
if off_prompt and on_prompt and abs(on_prompt - off_prompt) > max(8, 0.02 * off_prompt):
    print(
        f"Prompt length differs: off {off_prompt} tokens, on {on_prompt}. "
        "The prefill work is not the same."
    )
off_out = sum(row["completion_tokens"] for row in off_rows) / len(off_rows)
on_out = sum(row["completion_tokens"] for row in on_rows) / len(on_rows)
if abs(on_out - off_out) >= 2:
    print(
        f"Average answer length differs: off {off_out:.1f} tokens, on {on_out:.1f}. "
        "Figure 2 then mixes answer length with request time."
    )
if off_hits is not None and off_hits > 0:
    print(
        "The saved run recorded prefix-cache hits, so it is not a clean cache-off baseline. "
        "Restart with --no-enable-prefix-caching and re-run the previous code cell."
    )
off_first = off_rows[0]["ttft_s"]
if off_first > 0 and off_ttft < 0.5 * off_first:
    print(
        "In the saved run, requests after the first were much faster. "
        "That file does not look like prefix caching was off."
    )
on_first = on_rows[0]["ttft_s"]
if off_first > 0 and on_first < 0.5 * off_first:
    print(
        f"Request 1 time to first token is {on_first:.2f}s with caching on "
        f"and {off_first:.2f}s with it off. The prefix was probably already cached. "
        "Restart vLLM and run this cell again so request 1 is a cold prefill."
    )
if on_hits is not None and on_hits <= 0:
    print(
        "This server recorded no prefix-cache hits. "
        "It may still be the process started with --no-enable-prefix-caching."
    )
print("Saved outputs/prefix_caching.json")

COLORS = {ON: "#0B6E4F", OFF: "#1B4F9C"}
MARKERS = {ON: "o", OFF: "s"}

with plt.rc_context({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
    "font.size": 11,
    "axes.axisbelow": True,
}):
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 7.4), constrained_layout=True)
    fig.suptitle(f"Prefix caching on vs off — {model_name}", fontsize=14)

    def draw(ax, key, ylabel, title):
        for mode in (OFF, ON):
            rows = [row for row in prefix_rows if row["mode"] == mode]
            ax.plot(
                [row["request"] for row in rows],
                [row[key] for row in rows],
                label=mode,
                color=COLORS[mode],
                marker=MARKERS[mode],
                linewidth=2.25,
                markersize=8,
            )
        ax.set_title(title, loc="left", fontsize=12)
        ax.set_xlabel("Request number")
        ax.set_ylabel(ylabel)
        ax.set_xticks([row["request"] for row in on_rows])
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False)

    draw(
        axes[0],
        "ttft_s",
        "Seconds",
        "Figure 1. Time to first token (lower is faster)",
    )
    draw(
        axes[1],
        "tokens_per_s",
        "Output tokens per second",
        "Figure 2. Answer tokens / total request time",
    )

    fig.savefig("outputs/prefix_caching_tokens_per_s.png", dpi=140, bbox_inches="tight")
    plt.show()

print("Saved outputs/prefix_caching_tokens_per_s.png")


**Why Prefix Caching Matters**

Request 1 in each run pays the full prefill cost for the shared system prompt, so the time to first token matches. From request 2 onward, prefix caching reuses that prefix and the first token arrives much sooner. The run started with `--no-enable-prefix-caching` rebuilds the prefix on every call, so both curves stay near the first request.

Figure 1 is that time to first token. Figure 2 divides the short answer by the whole request, including prefill, so the shorter prefill shows up as a higher rate. Answers stay capped at a few tokens so generation does not hide the gap. The printed averages use requests 2 onward. The chart still draws request 1, which is the cold prefill on both runs.
